## Part 1 -- Groq Setup & Basic Chat

### Task 1 -- ChatGroq Setup

In [5]:
import os, time
from dotenv import load_dotenv
from langchain_groq import ChatGroq

In [6]:
load_dotenv()

python-dotenv could not parse statement starting at line 2


True

In [7]:
assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in .env"

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)


### Task 2 -- Basic Chat with ChatGroq

In [8]:
start = time.perf_counter()
response = llm.invoke("Explain in two sentences why Groq inference is fast.")
elapsed = time.perf_counter() - start

print(response.content)
print(f"Latency: {elapsed:.2f} s")

Groq inference is fast because its custom ASICs are built around a massively parallel, low‑latency architecture that processes entire tensors in a single, pipelined pass, eliminating the need for costly memory shuffling. Additionally, the system’s ultra‑low‑power, high‑bandwidth memory interface keeps data moving at gigabytes per second, so models run with minimal stalls and maximum throughput.
Latency: 0.60 s


## Part 2 -- RAG Backend Pipeline

### Task 3 -- Document Loading & Text Splittig

In [9]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\Acer\AppData\Local\Temp\ipykernel_18484\2838938482.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader


In [10]:
docs = []
docs += DirectoryLoader("data", glob="**/*.txt", loader_cls=TextLoader,loader_kwargs={"encoding": "utf-8"}).load()
docs += DirectoryLoader("data", glob="**/*.pdf", loader_cls=PyPDFLoader).load()
print(f"Loaded {len(docs)} document(s)")

Loaded 1 document(s)


In [12]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = splitter.split_documents(docs)
print(f"{len(docs)} document(s) -> {len(chunks)} chunks")
print("First chunk:", chunks[0].page_content)

1 document(s) -> 3 chunks
First chunk: Retrieval-Augmented Generation (RAG) is a technique that combines a retriever with a large language model. Instead of relying only on what the model memorised during training, a RAG system first searches an external knowledge base for passages relevant to the user's question, and then gives those passages to the model as context.


### Task 4 -- Embeddings & Vector Store

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

for d in retriever.invoke("What are chunk size and overlap?"):
    print("-", d.page_content[:100].replace("", " "), "...")

-  C h u n k   s i z e   a n d   c h u n k   o v e r l a p   c o n t r o l   h o w   d o c u m e n t s   a r e   s p l i t .   S m a l l   c h u n k s   g i v e   p r e c i s e   r e t r i e v a l   b u  ...
-  A   t y p i c a l   R A G   p i p e l i n e   h a s   t w o   s t a g e s .   I n   t h e   i n d e x i n g   s t a g e ,   d o c u m e n t s   a r e   l o a d e d ,   s p l i t   i n t o   s m a l l  ...
-  R e t r i e v a l - A u g m e n t e d   G e n e r a t i o n   ( R A G )   i s   a   t e c h n i q u e   t h a t   c o m b i n e s   a   r e t r i e v e r   w i t h   a   l a r g e   l a n g u a g e    ...


### Task 5 -- RAG Prompt Template

In [20]:
from langchain_core.prompts import ChatPromptTemplate

In [21]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant. Answer the question using ONLY the context below. ""If the answer is not in the context, say \"I don't know based on the provided context.\"""Context:{context}"),("human", "{question}"),])

question = "Why does chunk overlap matter?"
context = "".join(d.page_content for d in retriever.invoke(question))
print(rag_prompt.format(context=context, question=question))

System: You are a helpful assistant. Answer the question using ONLY the context below. If the answer is not in the context, say "I don't know based on the provided context."Context:Chunk size and chunk overlap control how documents are split. Small chunks give precise retrieval but may lose surrounding context, while large chunks keep more context but dilute relevance. Overlap repeats a little text between neighbouring chunks so that sentences cut at a boundary are not lost.

Groq provides very low-latency inference for open models through its LPU hardware, which makes it well suited as the generation step of a RAG application.A typical RAG pipeline has two stages. In the indexing stage, documents are loaded, split into smaller chunks, converted into vector embeddings and stored in a vector database such as FAISS or ChromaDB. In the query stage, the user's question is embedded, the most similar chunks are retrieved, and a prompt containing the question and the retrieved context is sent

## Part 3 -- ChatGroq RAG Chain

### Task 6 -- Build RAG Chain

In [22]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [23]:
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [24]:
questions = [
    "What is Retrieval-Augmented Generation?",
    "What are the two stages of a RAG pipeline?",
    "Why does chunk overlap matter?",
    "Which hardware does Groq use?",
    "Who won the 2018 FIFA World Cup?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}\n")

Q: What is Retrieval-Augmented Generation?
A: Retrieval‑Augmented Generation (RAG) is a technique that combines a retriever with a large language model. Instead of relying only on what the model memorised during training, a RAG system first searches an external knowledge base for passages relevant to the user’s question, and then gives those passages to the model as context.

Q: What are the two stages of a RAG pipeline?
A: The two stages of a Retrieval‑Augmented Generation (RAG) pipeline are:

1. **Indexing stage** – Documents are loaded, split into chunks, converted into vector embeddings, and stored in a vector database (e.g., FAISS or ChromaDB).

2. **Query stage** – The user’s question is embedded, the most similar chunks are retrieved from the vector database, and a prompt containing the question plus the retrieved context is sent to the language model.

Q: Why does chunk overlap matter?
A: Chunk overlap matters because it prevents sentences or ideas that fall right on a chunk bo

## Part 6 -- Mini Project Deliverable

The final app is `app.py` (run with `streamlit run app.py`). Multi-turn tests are in `test_chat.py`.

### Task 11 -- Observations & Insights

### Ques 1. Why Groq is suitable for RAG chatbots ?

#### Ans --> Groq runs open models on LPU hardware with very low latency and high tokens per second. A RAG request already spends time on retrieval, so a fast generation step keeps the whole chat feeling interactive. It is also cheap to run, so a large `k` or long context is practical.

### Ques 2. Difference between Groq RAG and OpenAI RAG ?
#### Ans --> The RAG pipeline (load, split, embed, retrieve, prompt) is identical. Only the generation model differs. Groq serves open models (Llama, GPT-OSS, Qwen) with very fast inference and a free tier, but the model choice is narrower and models are retired or renamed (our first model returned a 404). OpenAI offers proprietary models that are usually stronger at reasoning and instruction following, plus its own embeddings, but with higher latency and cost. Groq has no embeddings endpoint, so this project used local HuggingFace embeddings.

### Ques 3. Role of Streamlit in rapid GenAI prototyping ?

#### Ans --> Streamlit turns a Python script into a working web UI with no front-end code: `st.file_uploader`, `st.chat_input`, `st.chat_message` and `st.session_state` gave us a complete chat app in about 150 lines. It reruns the script on every interaction, so caching (`st.cache_resource`) is needed for the model and index. That is ideal for demos and prototypes, though not for large multi-user production apps.